In [1]:
from langchain_community.embeddings import OllamaEmbeddings
from langchain_community.llms import Ollama
from langchain_community.document_loaders import DirectoryLoader
from langchain_community.document_loaders.pdf import PyMuPDFLoader
from langchain.prompts import PromptTemplate
from langchain_community.vectorstores import FAISS
from langchain_community.vectorstores import DocArrayInMemorySearch
from langchain_text_splitters import CharacterTextSplitter
from langchain_text_splitters import TokenTextSplitter
from operator import itemgetter

In [19]:
import os

In [ ]:
Model = "llama3"; 
llmmodel = Ollama(model=Model)

template = """
Answer the question based on the page content in the context below. Make sure to check the source of the information in the metadata to ensure you pick the correct page_content. If you can't answer the question, reply with only "Oof that's a tough one, i don't really know this"

Context : {context}

Question : {question}

"""

prompt = PromptTemplate.from_template(template)

def format_docs(docs):
    return "\n\n".join(doc.page_content for doc in docs)

C:\Users\mannam\AppData\Local\Temp\ipykernel_25412\3819116841.py:2: LangChainDeprecationWarning: The class `Ollama` was deprecated in LangChain 0.3.1 and will be removed in 1.0.0. An updated version of the class exists in the :class:`~langchain-ollama package and should be used instead. To use it run `pip install -U :class:`~langchain-ollama` and import as `from :class:`~langchain_ollama import OllamaLLM``.
  llmmodel = Ollama(model=Model)


In [8]:
loader = DirectoryLoader(path="../../VectorStores",glob="**/*.pdf", loader_cls=PyMuPDFLoader)
documents = loader.load()

In [10]:
print(documents[0].page_content)

The Premier League is a professional association football league in England and the highest level of the 
English football league system. Contested by 20 clubs, it operates on a system of promotion and 
relegation with the English Football League (EFL). Seasons usually run from August to May, with each 
team playing 38 matches: two against each other team, one home and one away.[1] Most games are 
played on weekend afternoons, with occasional weekday evening fixtures.[2]  
  
The competition was founded as the FA Premier League on 20 February 1992, following the decision of 
First Division (the top-tier league from 1888 until 1992) clubs to break away from the English Football 
League. However, teams are still relegated to and promoted from the EFL Championship at the 
conclusion of each season. The Premier League is a corporation managed by a chief executive, with 
member clubs acting as shareholders.[3] The Premier League takes advantage of a £5 billion television 
rights deal, with 

In [17]:
text_splitter = TokenTextSplitter(chunk_size=400, chunk_overlap=0)
docs = text_splitter.split_documents(documents)
db = FAISS.from_documents(docs, OllamaEmbeddings(model = "nomic-embed-text"))

In [20]:
os.makedirs("../../VectorDB/nomic", exist_ok = True)
db.save_local("../../VectorDB/nomic/webMD")

In [21]:
retriever=db.as_retriever()

In [ ]:
chain = (
    {"context" : itemgetter("question") | retriever , "question" : itemgetter("question")}
    | prompt
    | llmmodel 
)

In [23]:
inquiry = "What is the premier league?"

In [24]:
print(chain.invoke({"question" : inquiry}))

Based on the provided content, the Premier League is a professional association football league in England that was formed in 1992. It consists of 20 teams and operates on a system of promotion and relegation with the English Football League Championship (EFL). The Premier League is one of the top four leagues in European football, along with the Spanish La Liga, Italian Serie A, and German Bundesliga.


In [25]:
inquiry = "Who has the most wins in the premier league?"

In [26]:
print(chain.invoke({"question" : inquiry}))

Based on the page content of the document with id '95b0d7c1-fd88-466f-902a-6e31ac0ee8a7', which corresponds to page 2, I can answer that:

"Most wins: Manchester United[1]"

This information is presented in the "Titles" section of the document.


In [28]:
print(chain.invoke({"question" : "who has won the games at most different stadiums?"}))

According to the page content of Document(id='527d45f6-33df-4454-87b1-e776bc1821f5'), the answer is:

"Most different stadiums won at: 59 (of 61), Liverpool"
